Before you turn this problem in, make sure everything runs as expected. First, **restart the kernel** (in the menubar, select Kernel$\rightarrow$Restart) and then **run all cells** (in the menubar, select Cell$\rightarrow$Run All).

Make sure you fill in any place that says `YOUR CODE HERE` or "YOUR ANSWER HERE", as well as your name and collaborators below:

NAME = "Nicole Paraschiv"
COLLABORATORS = ""

---

# CSCI 3155 Spring 2026: Assignment 7

Topics Covered: Function calls, closures, recursion and circular scopes.

In [2]:
// TEST HELPER
def passed(points: Int) = {
    require(points >=0)
    if (points == 1) print(s"\n*** Tests Passed (1 point) ***\n")
    else print(s"\n*** Tests Passed ($points points) ***\n")
}

defined function passed

## Problem 1 (50 Points): Mutual Recursion in Lettuce

In class, we have explored recursive functions in lettuce using the _let rec_ syntax. In this problem, we will 
explore, mutually recursive function, specifically two mutually recursive functions.

Consider: 

~~~
let rec 
        pos = function (x) 
                 if (x >= 0)
                 then x
                 else neg(x)
        neg = function (y)
                 if (y <= 0)
                 then pos (1 + y * y)
                 else pos(1 - y)
   in 
     neg(10.0)
~~~

The two functions are _mutually recursive_ since `pos` calls `neg` and vice-versa.
Convince yourself that the program must return `82`.

## 1A (10 points): Extending the Abstract Syntax

Consider the grammar specification we have seen thus far.

$$\begin{array}{rcll}
\mathbf{Program} & \rightarrow & TopLevel(\mathbf{Expr}) \\[5pt]
\mathbf{Expr} & \rightarrow & Const(\mathbf{Number}) \\
 & | & Ident(\mathbf{Identifier}) \\
 & | & Plus(\mathbf{Expr}, \mathbf{Expr}) \\
 & | & Mult(\mathbf{Expr}, \mathbf{Expr}) \\
 & | & Eq(\mathbf{Expr}, \mathbf{Expr}) \\
 & | & Geq (\mathbf{Expr}, \mathbf{Expr}) \\
 & | & IfThenElse(\mathbf{Expr}, \mathbf{Expr}, \mathbf{Expr}) & \text{if (expr) then expr else expr} \\
 & | & Let( \mathbf{Identifier}, \mathbf{Expr}, \mathbf{Expr}) & \text{let identifier = expr in expr} \\
 & | & FunDef( \mathbf{Identifier}, \mathbf{Expr}) & \text{function (identifier-formal-parameter) expr } \\ 
 & | & FunCall(\mathbf{Expr}, \mathbf{Expr}) & \text{function call - expr(expr)} \\
 & | & LetRec(\mathbf{Identifier}, \mathbf{Identifier}, \mathbf{Expr}, \mathbf{Expr})  \\
\end{array}$$

We wish to add a new rule for two mutually recursive functions

$$ \mathbf{Expr}\ \rightarrow\ LetRec2( \mathbf{Identifier}, \mathbf{Identifier}, \mathbf{Expr}, \mathbf{Identifier}, \mathbf{Identifier}, \mathbf{Expr}, \mathbf{Expr} ) $$

Such that a mutual call such as

~~~
let rec 
         f1 = function (x1) e1
         f2 = function (x2) e2
     in 
         e3
~~~
is represented in the AST as

~~~
LetRec2(f1, x1, e1, f2, x2, e2, e3)
~~~

Extend the existing AST specification to add support for `LetRec2`.


In [3]:
sealed trait Program
sealed trait Expr
case class Const(f: Double) extends Expr
case class Ident(s: String) extends Expr
case class Minus(e1: Expr, e2: Expr) extends Expr
case class Plus(e1: Expr, e2: Expr) extends Expr
case class Mult(e1: Expr, e2: Expr) extends Expr
case class Eq(e1: Expr, e2: Expr) extends Expr
case class Geq(e1: Expr, e2: Expr) extends Expr
case class IfThenElse(e1: Expr, e2: Expr, e3: Expr) extends Expr
case class Let(x: String, e1: Expr, e2: Expr) extends Expr
case class FunDef(id: String, e: Expr) extends Expr
case class FunCall(calledFun: Expr, argExpr: Expr) extends Expr
case class LetRec(funName: String, param: String, funExpr: Expr, bodyExpr: Expr) extends Expr

// YOUR CODE HERE
case class LetRec2(f1: String, x1: String, e1: Expr, f2: String, x2: String, e2: Expr, e3: Expr) extends Expr
case class TopLevel(e: Expr) extends Program

defined trait Program
defined trait Expr
defined class Const
defined class Ident
defined class Minus
defined class Plus
defined class Mult
defined class Eq
defined class Geq
defined class IfThenElse
defined class Let
defined class FunDef
defined class FunCall
defined class LetRec
defined class LetRec2
defined class TopLevel

In [4]:
//BEGIN TEST
val x = Ident("x")
val y = Ident("y")
val foo = Ident("foo")
val bar = Ident("bar")

val e1 = IfThenElse( Geq(x, Const(0.0)), x, FunCall(bar, Plus(x, Const(1.0)))) // if x >= 0 then x else bar(1 + x)
val e2 = IfThenElse( Geq(Const(1.0), x), Plus(Const(2.0), x), FunCall(foo, Minus(x, Const(2.0)))) // if 1 >= x then 2 + x else foo(x-2)
val e3 = FunCall(bar, Const(10))
val lr2 = LetRec2("foo", "x", e1, "bar", "x", e2, e3)
val p1 = TopLevel(lr2)
passed(5)
//END TEST


*** Tests Passed (5 points) ***


x: Ident = Ident(s = "x")
y: Ident = Ident(s = "y")
foo: Ident = Ident(s = "foo")
bar: Ident = Ident(s = "bar")
e1: IfThenElse = IfThenElse(
  e1 = Geq(e1 = Ident(s = "x"), e2 = Const(f = 0.0)),
  e2 = Ident(s = "x"),
  e3 = FunCall(
    calledFun = Ident(s = "bar"),
    argExpr = Plus(e1 = Ident(s = "x"), e2 = Const(f = 1.0))
  )
)
e2: IfThenElse = IfThenElse(
  e1 = Geq(e1 = Const(f = 1.0), e2 = Ident(s = "x")),
  e2 = Plus(e1 = Const(f = 2.0), e2 = Ident(s = "x")),
  e3 = FunCall(
    calledFun = Ident(s = "foo"),
    argExpr = Minus(e1 = Ident(s = "x"), e2 = Const(f = 2.0))
  )
)
e3: FunCall = FunCall(calledFun = Ident(s = "bar"), argExpr = Const(f = 10.0))
lr2: LetRec2 = LetRec2(
  f1 = "foo",
  x1 = "x",
  e1 = IfThenElse(
    e1 = Geq(e1 = Ident(s = "x"), e2 = Const(f = 0.0)),
    e2 = Ident(s = "x"),
    e3 = FunCall(
      calledFun = Ident(s = "bar"),
      argExpr = Plus(e1 = Ident(s = "x"), e2 = Const(f = 1.0))
    )
  ),
  f2 = "bar",
  x2 = "x",
  e2 = IfThenElse(
    e1 

In [5]:
//BEGIN TEST
val x = Ident("x")
val y = Ident("y")
val foo = Ident("foo")
val bar = Ident("bar")
val e11 = IfThenElse(Geq(x, Const(0.0)), FunCall(bar, Minus(Const(1.0), x)),Mult(x, FunCall(foo,  Minus(x, Const(1.0))) ))
val e1 = IfThenElse(Eq(x, Const(0.0)), Const(1.0), e11)
val e2 = IfThenElse(Geq(Const(0.0), y), Mult(Const(-0.5), y) , FunCall(foo, y))
val e3 = FunCall(foo, Const(10.5))
val lr3 = LetRec2("foo", "x", e1, "bar", "y", e2, e3)
passed(5)
//END TEST


*** Tests Passed (5 points) ***


x: Ident = Ident(s = "x")
y: Ident = Ident(s = "y")
foo: Ident = Ident(s = "foo")
bar: Ident = Ident(s = "bar")
e11: IfThenElse = IfThenElse(
  e1 = Geq(e1 = Ident(s = "x"), e2 = Const(f = 0.0)),
  e2 = FunCall(
    calledFun = Ident(s = "bar"),
    argExpr = Minus(e1 = Const(f = 1.0), e2 = Ident(s = "x"))
  ),
  e3 = Mult(
    e1 = Ident(s = "x"),
    e2 = FunCall(
      calledFun = Ident(s = "foo"),
      argExpr = Minus(e1 = Ident(s = "x"), e2 = Const(f = 1.0))
    )
  )
)
e1: IfThenElse = IfThenElse(
  e1 = Eq(e1 = Ident(s = "x"), e2 = Const(f = 0.0)),
  e2 = Const(f = 1.0),
  e3 = IfThenElse(
    e1 = Geq(e1 = Ident(s = "x"), e2 = Const(f = 0.0)),
    e2 = FunCall(
      calledFun = Ident(s = "bar"),
      argExpr = Minus(e1 = Const(f = 1.0), e2 = Ident(s = "x"))
    ),
    e3 = Mult(
      e1 = Ident(s = "x"),
      e2 = FunCall(
        calledFun = Ident(s = "foo"),
        argExpr = Minus(e1 = Ident(s = "x"), e2 = Const(f = 1.0))
      )
    )
  )
)
e2: IfThenElse = IfThenElse(

## 1B (20 points): Build an Environment to Handle Mutual Recursion

In class and notes, we saw how to build an environment that handles recursive calls using the `ExtendRec` construct.

Now we propose an `ExtendMutualRec2` construct such that

$$\begin{array}{c}
\\
\hline
{ \text{eval}(\texttt{LetRec2(f1, x1, e1, f2, x2, e2, e)}, \sigma) = \text{eval}(e, \texttt{ExtendMutualRec2}(\texttt{f1, x1, e1, f2, x2, e2}, \sigma)) } \\
\end{array}(\text{Mutual-Rec-OK})$$

To implement mutual recursion:

* The `Environment` needs one more case class `ExtendMutualRec2`, which will be used to implement `LetRec2` as shown above.
* The `lookupEnv` function needs to handle 4 cases: `EmptyEnv`, `Extend`, `ExtendRec`, and `ExtendMutualRec2`. The first three cases are implemented exactly as in the lecture notes. The `ExtendMutualRec2` case should return one of the two possible closures depending on which of the two mutually recursive functions is being looked up. 

Complete the definition of the `Environment` and `lookupEnv` using the logic described above.

In [6]:
sealed trait Environment 
sealed trait Value

case object EmptyEnv extends Environment
case class Extend(x: String, v: Value, sigma: Environment) extends Environment
case class ExtendRec(f: String, x: String, e: Expr, sigma: Environment ) extends Environment
case class ExtendMutualRec2(
    // YOUR CODE HERE
    f1: String,
    x1: String,
    e1: Expr,
    f2: String,
    x2: String,
    e2: Expr,
    sigma: Environment
) extends Environment

/* -- We need to redefine values to accomodate the new representation of environments --*/
case class NumValue(d: Double) extends Value
case class BoolValue(b: Boolean) extends Value
case class Closure(x: String, e: Expr, pi: Environment) extends Value
case object ErrorValue extends Value


/*2. Operators on values */

def valueToNumber(v: Value): Double = v match {
    case NumValue(d) => d
    case _ => throw new IllegalArgumentException(s"Error: Asking me to convert Value: $v to a number")
}

def valueToBoolean(v: Value): Boolean = v match {
    case BoolValue(b) => b
    case _ => throw new IllegalArgumentException(s"Error: Asking me to convert Value: $v to a boolean")
}

def valueToClosure(v: Value): Closure = v match {
    case Closure(x, e, pi) => Closure(x, e, pi)
    case _ =>  throw new IllegalArgumentException(s"Error: Asking me to convert Value: $v to a closure")
}


/*-- Operations on environments --*/

def lookupEnv(pi: Environment, x: String): Value = pi match {
    // YOUR CODE HERE
    case EmptyEnv => throw new IllegalArgumentException("Empty")
    case Extend(y, v, sigma) => {
        if (x==y) v
        else lookupEnv(sigma, x)
    }
    case ExtendRec(f, y, e, sigma) => {
        if(f == x) Closure(y, e, pi)
        else lookupEnv(sigma, x)
    }
    case ExtendMutualRec2(f1, x1, e1, f2, x2, e2, sigma) => {
        if(f1 == x) Closure(x1, e1, pi)
        else if (f2 == x) Closure(x2, e2, pi)
        else lookupEnv(sigma, x)
    }
}

defined trait Environment
defined trait Value
defined object EmptyEnv
defined class Extend
defined class ExtendRec
defined class ExtendMutualRec2
defined class NumValue
defined class BoolValue
defined class Closure
defined object ErrorValue
defined function valueToNumber
defined function valueToBoolean
defined function valueToClosure
defined function lookupEnv

In [7]:
// BEGIN TEST
val env: Environment = ExtendMutualRec2("f1", "x1", Const(0.0), "f2", "x2", Const(0.0), EmptyEnv)
passed(5)
// END TEST


*** Tests Passed (5 points) ***


env: Environment = ExtendMutualRec2(
  f1 = "f1",
  x1 = "x1",
  e1 = Const(f = 0.0),
  f2 = "f2",
  x2 = "x2",
  e2 = Const(f = 0.0),
  sigma = EmptyEnv
)

In [8]:
// BEGIN TEST
val env: Environment = ExtendMutualRec2("f1", "x1", Const(0.0), "f2", "x2", Const(0.0), EmptyEnv)
// Ensure looking up either function gets us the right value, no matter how many times we recurse.
val f1 @ Closure(_, _, pi1) = lookupEnv(env, "f1")
val f2 @ Closure(_, _, pi2) = lookupEnv(env, "f2")
lookupEnv(pi1, "f1") == f1
lookupEnv(pi1, "f2") == f2
lookupEnv(pi2, "f2") == f1
lookupEnv(pi2, "f2") == f2
passed(15)
// END TEST

-- Warning: cmd8.sc:3:39 -------------------------------------------------------
3 |val f1 @ Closure(_, _, pi1) = lookupEnv(env, "f1")
  |                              ^^^^^^^^^^^^^^^^^^^^
  |pattern's type cmd8.this.cmd6.Closure is more specialized than the right hand side expression's type cmd8.this.cmd6.Value
  |
  |If the narrowing is intentional, this can be communicated by adding `: @unchecked` after the expression,
  |which may result in a MatchError at runtime.
-- Warning: cmd8.sc:4:39 -------------------------------------------------------
4 |val f2 @ Closure(_, _, pi2) = lookupEnv(env, "f2")
  |                              ^^^^^^^^^^^^^^^^^^^^
  |pattern's type cmd8.this.cmd6.Closure is more specialized than the right hand side expression's type cmd8.this.cmd6.Value
  |
  |If the narrowing is intentional, this can be communicated by adding `: @unchecked` after the expression,
  |which may result in a MatchError at runtime.



*** Tests Passed (15 points) ***


env: Environment = ExtendMutualRec2(
  f1 = "f1",
  x1 = "x1",
  e1 = Const(f = 0.0),
  f2 = "f2",
  x2 = "x2",
  e2 = Const(f = 0.0),
  sigma = EmptyEnv
)
res8_3: Boolean = true
res8_4: Boolean = true
res8_5: Boolean = false
res8_6: Boolean = true

## 1C (20 points): Interpreter

Fill in the missing parts of the interpreter for the extended language. You will have to implement the cases for:
* `FunDef`
* `FunCall`
* `LetRec`
* `LetRec2`

In [9]:
def evalExpr(e: Expr, env: Environment): Value =  {
    
    /* Method to deal with binary arithmetic operations */
    def applyArith2 (e1: Expr, e2: Expr) (fun: (Double , Double) => Double) = {
        val v1 = valueToNumber(evalExpr(e1, env))
        val v2 = valueToNumber(evalExpr(e2, env))
        val v3 = fun(v1, v2)
        NumValue(v3)
    }  /* -- We have deliberately curried the method --*/
    
    /* Helper method to deal with unary arithmetic */
    def applyArith1(e: Expr) (fun: Double => Double) = {
        val v = valueToNumber(evalExpr(e, env))
        val v1 = fun(v)
        NumValue(v1)
    }
    
    /* Helper method to deal with comparison operators */
    def applyComp(e1: Expr, e2: Expr) (fun: (Double, Double) => Boolean) = {
        val v1 = valueToNumber(evalExpr(e1, env))
        val v2 = valueToNumber(evalExpr(e2, env))
        val v3 = fun(v1, v2)
        BoolValue(v3)
    }
    
   
    e match {
        case Const(f) => NumValue(f) // Same as before
        
        case Ident(x) => lookupEnv(env, x) // Changed to accomodate the new environment definitions.
    
        
        case Plus(e1, e2) => applyArith2 (e1, e2) ( _ + _ )
        
        case Minus(e1, e2) => applyArith2(e1, e2) ( _ - _ )
        
        case Mult(e1, e2) =>  applyArith2(e1, e2) (_ * _)
        
        case Geq(e1, e2) => applyComp(e1, e2)(_ >= _)
        
        case Eq(e1, e2) => applyComp(e1, e2)(_ == _)
        
        case IfThenElse(e1, e2, e3) => {
            val v = evalExpr(e1, env)
            v match {
                case BoolValue(true) => evalExpr(e2, env)
                case BoolValue(false) => evalExpr(e3, env)
                case _ => throw new IllegalArgumentException(s"If-then-else condition Expr: ${e1} is non-boolean -- evaluates to ${v}")
            }
        }
        
        case Let(x, e1, e2) => {
            val v1 = evalExpr(e1, env)  // eval e1
            val env2 = Extend(x, v1, env) // create a new extended env
            evalExpr(e2, env2) // eval e2 under that.
        }
        
        case FunDef(x, e) => {
            // YOUR CODE HERE
            Closure(x, e, env)
        }
        
        case FunCall(e1, e2) => {
            val v1 = evalExpr(e1, env)
            val v2 = evalExpr(e2, env)
            v1 match {
                // YOUR CODE HERE
                case Closure(x, closure_ex, closed_env) => {
                    // First extend closed_env by binding x to v2
                    val new_env = Extend(x, v2, closed_env)
                    // Evaluate the body of the closure under the extended environment.
                    evalExpr(closure_ex, new_env)
                }
                case _ => throw new IllegalArgumentException(s"Function call error: expression $e1 does not evaluate to a closure")
            }
        }
    
        case LetRec(rfun, x, fExpr, bExpr) => {
            // YOUR CODE HERE
            /* let rec f = function(x) e1 in e2 meaning function f can call itself inside e1
            f is bound in the body of e2 and f is also visible inside its own function body e1 { f can call itself */
            val rEnv = ExtendRec(rfun, x, fExpr, env) //<- new ENv
            evalExpr(bExpr, rEnv)
        }
        
        case LetRec2(f1, x1, e1, f2, x2, e2, e) => {
            // YOUR CODE HERE
            /*  we want to support two functions calling each other:
                let rec 
                    f1 = function(x1) e1
                    f2 = function(x2) e2
                in
                    e3 (as shown above)
            This means f1 can call f2 and f2 can call f1 { they depend on each other
            in mutual recursion we must have both calling each other and themselves 
            We cant to create an environemnt where both functions exist
            each function closure must point to the same environment
            then we evaluate e3 
            */
            val rEnv = ExtendMutualRec2(f1,x1,e1,f2,x2,e2,env) // <- new env for this
            evalExpr(e,rEnv)
        }
    }
}

def evalProgram(p: Program) = {
    p match { 
        case TopLevel(e) => evalExpr(e, EmptyEnv)
    }
}


defined function evalExpr
defined function evalProgram

In [10]:
//BEGIN TEST
val x = Ident("x")
val y = Ident("y")
val foo = Ident("foo")
val bar = Ident("bar")

val e1 = IfThenElse( Geq(x, Const(0.0)), x, FunCall(bar, Plus(x, Const(1.0)))) // if x >= 0 then x else bar(1 + x)
val e2 = IfThenElse( Geq(Const(1.0), x), Plus(Const(2.0), x), FunCall(foo, Minus(x, Const(2.0)))) // if 1 >= x then 2 + x else foo(x-2)
val e3 = FunCall(bar, Const(10))

/* -- 
let factorial = function (n) 
            if (n == 0) 
            then 1
            else n * factorial(n-1)
   in 
    factorial(10)
--*/
val fact = Ident("factorial")
val n = Ident("n")
// if n == 0 then 1 else n * factorial(n-1)
val ite = IfThenElse(Eq(n, Const(0)), Const(1), Mult(n, FunCall(fact, Minus(n, Const(1)))))
val e = LetRec("factorial", "n", ite, FunCall(fact, Const(5)))
assert(evalProgram(TopLevel(e)) == NumValue(120.0), "Test 0 of Set 1 failed")

val lr2 = LetRec2("foo", "x", e1, "bar", "x", e2, e3)
val p1 = TopLevel(lr2)
assert(evalProgram(p1) == NumValue(8.0), "Test 1 of Set 1 failed")

val e4 = FunCall(foo, Const(12.0))
val p2 = TopLevel(LetRec2("foo", "x", e1, "bar", "x", e2, e4))
assert(evalProgram(p2) == NumValue(12.0), "Test 2 of Set 1 failed")

val e5 = FunCall(foo, Const(-12.0))
val p3 = TopLevel(LetRec2("foo", "x", e1, "bar", "x", e2, e5))
assert(evalProgram(p3) == NumValue(-9.0), "Test 3 of Set 1 failed")

val e6 = FunCall(bar, Const(-12.0))
val p4 = TopLevel(LetRec2("foo", "x", e1, "bar", "x", e2, e6))
assert(evalProgram(p4) == NumValue(-10.0), "Test 4 of Set 1 failed")

val e7 = FunCall(bar, Const(1.9))
val p5 = TopLevel(LetRec2("foo", "x", e1, "bar", "x", e2, e7))
assert(evalProgram(p5) == NumValue(2.9), "Test 5 of Set 1 failed")

passed(10)
//END TEST


*** Tests Passed (10 points) ***


x: Ident = Ident(s = "x")
y: Ident = Ident(s = "y")
foo: Ident = Ident(s = "foo")
bar: Ident = Ident(s = "bar")
e1: IfThenElse = IfThenElse(
  e1 = Geq(e1 = Ident(s = "x"), e2 = Const(f = 0.0)),
  e2 = Ident(s = "x"),
  e3 = FunCall(
    calledFun = Ident(s = "bar"),
    argExpr = Plus(e1 = Ident(s = "x"), e2 = Const(f = 1.0))
  )
)
e2: IfThenElse = IfThenElse(
  e1 = Geq(e1 = Const(f = 1.0), e2 = Ident(s = "x")),
  e2 = Plus(e1 = Const(f = 2.0), e2 = Ident(s = "x")),
  e3 = FunCall(
    calledFun = Ident(s = "foo"),
    argExpr = Minus(e1 = Ident(s = "x"), e2 = Const(f = 2.0))
  )
)
e3: FunCall = FunCall(calledFun = Ident(s = "bar"), argExpr = Const(f = 10.0))
fact: Ident = Ident(s = "factorial")
n: Ident = Ident(s = "n")
ite: IfThenElse = IfThenElse(
  e1 = Eq(e1 = Ident(s = "n"), e2 = Const(f = 0.0)),
  e2 = Const(f = 1.0),
  e3 = Mult(
    e1 = Ident(s = "n"),
    e2 = FunCall(
      calledFun = Ident(s = "factorial"),
      argExpr = Minus(e1 = Ident(s = "n"), e2 = Const(f = 1.0))

In [11]:
//BEGIN TEST 
/*
let rec 
        pos = function (x) 
                 if (x >= 0)
                 then
                    if (x <= 0.5)
                     then x
                     else 0.5 + pos(x-1)
                 else neg(x)
        neg = function (y)
                 if (y <= 0)
                 then pos (1 + y * y)
                 else pos(1 - y)
   in 
     ...
*/

val x = Ident("x")
val y = Ident("y")
val pos = Ident("pos")
val neg = Ident("neg")

val e11 = IfThenElse(Geq(Const(0.5), x), x, Plus(Const(0.5), FunCall(pos, Minus(x, Const(1.0)))))
val e1 = IfThenElse(Geq(x, Const(0.0)), e11, FunCall(neg, x))
val e2 = IfThenElse(Geq(Const(0.0), y), FunCall(pos, Plus(Const(1.0), Mult(y,y))), FunCall(pos, Minus(Const(1.0), y)))

val t1 = FunCall(neg, Const(10.0))
val lr1 = LetRec2("pos", "x", e1, "neg", "y", e2, t1)
val p1 = TopLevel(lr1)
assert(evalProgram(p1) == NumValue(41.0), "Test 1 of set 2 failed")

val t2 = FunCall(pos, Const(-8.0))
val lr2 = LetRec2("pos", "x", e1, "neg", "y", e2, t2)
val p2 = TopLevel(lr2)
assert(evalProgram(p2) == NumValue(32.5), "Test 2 of set 2 failed")


val t3 = FunCall(pos, Const(-0.5))
val lr3 = LetRec2("pos", "x", e1, "neg", "y", e2, t3)
val p3 = TopLevel(lr3)
assert(evalProgram(p3) == NumValue(0.75), "Test 3 of set 2 failed")

passed(10)
//END TEST


*** Tests Passed (10 points) ***


x: Ident = Ident(s = "x")
y: Ident = Ident(s = "y")
pos: Ident = Ident(s = "pos")
neg: Ident = Ident(s = "neg")
e11: IfThenElse = IfThenElse(
  e1 = Geq(e1 = Const(f = 0.5), e2 = Ident(s = "x")),
  e2 = Ident(s = "x"),
  e3 = Plus(
    e1 = Const(f = 0.5),
    e2 = FunCall(
      calledFun = Ident(s = "pos"),
      argExpr = Minus(e1 = Ident(s = "x"), e2 = Const(f = 1.0))
    )
  )
)
e1: IfThenElse = IfThenElse(
  e1 = Geq(e1 = Ident(s = "x"), e2 = Const(f = 0.0)),
  e2 = IfThenElse(
    e1 = Geq(e1 = Const(f = 0.5), e2 = Ident(s = "x")),
    e2 = Ident(s = "x"),
    e3 = Plus(
      e1 = Const(f = 0.5),
      e2 = FunCall(
        calledFun = Ident(s = "pos"),
        argExpr = Minus(e1 = Ident(s = "x"), e2 = Const(f = 1.0))
      )
    )
  ),
  e3 = FunCall(calledFun = Ident(s = "neg"), argExpr = Ident(s = "x"))
)
e2: IfThenElse = IfThenElse(
  e1 = Geq(e1 = Const(f = 0.0), e2 = Ident(s = "y")),
  e2 = FunCall(
    calledFun = Ident(s = "pos"),
    argExpr = Plus(
      e1 = Const(f 

### That's all folks